# STEP 1: Overview (Markdown Description)
# --------------------------------------
# This script converts a daily ground temperature dataset into a half-hourly dataset.
#
# INPUT:
# - CSV file with 365 rows (one per day) containing:
#     - Time (ignored except for ordering)
#     - ground_temp_30mm (daily temperature value)
#
# OUTPUT:
# - CSV file with 17,520 rows (365 days × 48 half-hour intervals per day)
# - Columns:
#     - Time (HH:MM:SS format, starting from 00:30:00, no dates)
#     - ground_temp_30mm (same value repeated 48 times per day)
#
# PROCESS:
# 1. Load the daily dataset
# 2. Repeat each daily temperature value 48 times
# 3. Generate a repeating half-hour time sequence (00:30 → 24:00 looped daily)
# 4. Combine into a new dataframe
# 5. Save to specified output path

In [20]:
# STEP 2: Import Required Libraries
# --------------------------------
import pandas as pd

In [21]:
# STEP 3: Load the Input CSV
# -------------------------
input_path = '/Users/jakegehrung/Desktop/data_raw/REFERENCE FILES/daily_ground_temp.csv'

df = pd.read_csv(input_path)

# Check structure
print(df.head())
print(f"Number of rows in input: {len(df)}")

           Time  ground_temp_30mm
0  1/1/19 09:00               6.0
1  1/2/19 09:00               6.2
2  1/3/19 09:00               5.5
3  1/4/19 09:00               5.5
4  1/5/19 09:00               5.5
Number of rows in input: 365


In [22]:
# STEP 4 (FIXED): Clean and Validate Temperature Column
# ----------------------------------------------------
# This step ensures:
# - No missing values
# - Correct column name
# - Exactly 365 usable entries

# Strip column names (handles hidden spaces)
df.columns = df.columns.str.strip()

# Convert temperature column to numeric (forces errors to NaN)
temps = pd.to_numeric(df['ground_temp_30mm'], errors='coerce')

# Drop any NaN values (in case of bad rows)
temps = temps.dropna().reset_index(drop=True)

print(f"Valid temperature rows after cleaning: {len(temps)}")

# Check if we still have 365 days
if len(temps) != 365:
    raise ValueError(f"Expected 365 valid temperature values, but got {len(temps)}. Check your CSV for missing or malformed data.")

Valid temperature rows after cleaning: 365


In [23]:
# STEP 5: Expand Each Day to 48 Half-Hour Values
# ---------------------------------------------
expanded_temps = temps.repeat(48).reset_index(drop=True)

print(f"Expanded rows: {len(expanded_temps)}")  # Should be 17520

Expanded rows: 17520


In [24]:
# STEP 6 (FIXED): Generate EXACTLY 48 Half-Hour Time Steps
# ------------------------------------------------------
# Your previous version produced the wrong number of intervals.

import pandas as pd

# Generate exactly 48 half-hour intervals starting at 00:30
time_range = pd.date_range(
    start="00:30:00",
    periods=48,
    freq="30min"
)

# Convert to HH:MM:SS format
time_strings = time_range.strftime("%H:%M:%S")

# Repeat for 365 days
full_time_series = list(time_strings) * 365

print(f"Time entries: {len(full_time_series)}")  # Should be 17520

Time entries: 17520


In [25]:
# STEP 7 (SAFE COMBINE WITH CHECK)
# -------------------------------
# Final safety check before combining

print(f"Time length: {len(full_time_series)}")
print(f"Temp length: {len(expanded_temps)}")

assert len(full_time_series) == len(expanded_temps), "Mismatch between time and temperature lengths!"

output_df = pd.DataFrame({
    'Time': full_time_series,
    'ground_temp_30mm': expanded_temps
})

print(output_df.head())

Time length: 17520
Temp length: 17520
       Time  ground_temp_30mm
0  00:30:00               6.0
1  01:00:00               6.0
2  01:30:00               6.0
3  02:00:00               6.0
4  02:30:00               6.0


In [26]:
# STEP 8: Save to Output CSV
# -------------------------
output_path = '/Users/jakegehrung/Desktop/data_raw/REFERENCE FILES/half_hourly_ground_temp.csv'

output_df.to_csv(output_path, index=False)

print(f"File saved successfully to:\n{output_path}")

File saved successfully to:
/Users/jakegehrung/Desktop/data_raw/REFERENCE FILES/half_hourly_ground_temp.csv
